# OBO / OWL ontologies — API & bulk

`biodb.ontology` covers the entire OBO Foundry — Mondo, HPO, EFO, SO,
GO, and any other OBO/OWL file.

| Mode | Functions | What it does |
|---|---|---|
| **API (owlready2 entity walk)** | `get_ontology`, `get_descendants`, `get_ancestors`, `get_mrca` | Point-lookup traversal on a loaded ontology object |
| **Bulk (download + parse)** | `expand_keyword_sets_from_ontology`, `download_ontology`, `parse_obo` | Whole-graph keyword expansion, attention-weight analysis, ontology-similarity matrices |

Requires the `[ontology]` extra (`owlready2`, `obonet`, `pronto`, …).

## 1. Generic OWL entity walk

`get_ontology` loads any OBO Foundry OWL file via `owlready2`. The
returned object exposes every class — you can walk it freely.

In [1]:
from biodb import ontology

(Skipped at notebook-execute time — owlready2 download is slow.
Uncomment to run against the real Mondo OWL release.)

```python
mondo = ontology.get_ontology("http://purl.obolibrary.org/obo/mondo.owl").load()
alzheimers = mondo.search_one(iri="*MONDO_0004975")
descendants = ontology.get_descendants(alzheimers)
print(f"{alzheimers.label[0]} has {len(descendants)} descendants")
```

## 2. N-hop keyword expansion over an in-memory graph

`expand_keyword_sets_from_ontology` takes a seed set and walks an
in-memory ontology graph N hops outward. Works against any
`{parent: [children]}` dict — typically built from a parsed Mondo /
HPO release.

In [2]:
ontology_graph = {
    "dementia": ["alzheimer's disease", "vascular dementia", "lewy body dementia"],
    "alzheimer's disease": ["early-onset alzheimer's", "late-onset alzheimer's"],
    "vascular dementia": ["multi-infarct dementia"],
    "lewy body dementia": [],
    "early-onset alzheimer's": [],
    "late-onset alzheimer's": [],
    "multi-infarct dementia": [],
}
seeds = {"dementia": ["dementia"]}

In [3]:
ontology.expand_keyword_sets_from_ontology(
    seed_keywords=seeds, ontology_dict=ontology_graph, n_hops=1
)

{'dementia': ['dementia',
  "alzheimer's disease",
  'lewy body dementia',
  'vascular dementia']}

In [4]:
ontology.expand_keyword_sets_from_ontology(
    seed_keywords=seeds, ontology_dict=ontology_graph, n_hops=2
)

{'dementia': ['dementia',
  "alzheimer's disease",
  'lewy body dementia',
  'vascular dementia',
  "early-onset alzheimer's",
  "late-onset alzheimer's",
  'multi-infarct dementia']}

## 3. Bulk ontology download + parse

For Mondo / HPO / EFO downloads, use the OBO Foundry PURLs directly:

```python
from biodb import ontology

# Load + parse the full Mondo OBO release (~30 MB):
mondo_graph = ontology.parse_obo("http://purl.obolibrary.org/obo/mondo.obo")
# Or via owlready2 for OWL-level semantics:
mondo_owl = ontology.get_ontology("http://purl.obolibrary.org/obo/mondo.owl").load()
```

Once loaded, all the entity-walk helpers (`get_descendants`,
`get_ancestors`, `get_mrca`) work directly on the loaded objects.